# 공공 자전거 대여 데이터 파이프라인

---

## 배경

가상의 공공 자전거 사업자 **BikeCity** 의 데이터입니다.

운영팀이 넘겨준 CSV 3개를 정제해 분석용 DB 를 구축하는 것이 여러분의 일입니다.

🗣️ **운영팀 담당자의 말**

> "시스템 이전을 두 번 했는데 그때마다 형식이 좀 달라졌어요. 그리고 작년에 배치 스크립트가 오작동해서 같은 대여 기록이 중복으로 들어간 적 있습니다."
> 

---

## 제공 데이터

data.zip

| 파일 | 행 수 | 내용 |
| --- | --- | --- |
| `stations.csv` | 30 | 대여소 마스터 (깨끗함) |
| `raw-bikes.csv` | 55 | 자전거 마스터 (오염됨) |
| `raw-rentals.csv` | 15,000 | 대여·반납 기록 (오염됨) |

### 컬럼 설명

#### **stations.csv**

`station_id` 대여소 ID (S01~S30) · `station_name` 대여소명 · `district` 자치구

#### **raw-bikes.csv**

`bike_id` 자전거 ID · `station_id` 배치 대여소 · `bike_type` 자전거 종류

`gear_count` 기어 수 · `manufacture_year` 제조연도 · `daily_fee` 1일 기본요금(원)

#### **raw-rentals.csv**

`rental_id` 대여 기록 ID · `bike_id` 자전거 ID · `user_id` 이용자 ID

`rent_time` 대여 시각 · `return_time` 반납 시각 · `distance_km` 이동 거리

`fee` 요금(원) · `payment_method` 결제수단

> 작업 폴더에 `data/` 를 만들고 세 파일을 넣은 뒤 시작하세요.
>

In [1]:
import pandas as pd

from etl.config import raw_bikes_path, raw_rentals_path, stations_path, ENCODING

## STEP 1 · 진단

정제하기 전에 **무엇이 얼마나 망가졌는지** 파악합니다.

세 파일을 읽고 아래를 확인해 표로 정리하세요.

- 각 파일의 행 수, 컬럼별 dtype
- 숫자여야 하는데 문자열로 읽힌 컬럼
- 컬럼별 결측 수와 비율
- `bike_id`, `rental_id` 의 중복 건수
- `bike_type`, `district`, `payment_method` 의 고유값 목록

<aside>
💡

**참고**

`dtype=str, keep_default_na=False` 로 읽으면 파일에 적힌 그대로 볼 수 있습니다.

`read_csv` 는 `"N/A"` 를 자동으로 결측 처리하므로, 원본 상태를 보려면 이 옵션이 필요합니다.

</aside>

### 확인 문항

1. `raw-bikes.csv` 는 55행인데 자전거는 몇 대입니까? 왜 다릅니까?
2. `bike_type` 의 고유값은 몇 종류입니까? 실제로는 몇 종류여야 합니까?
3. `distance_km` 을 숫자로 못 바꾸는 값에는 어떤 것들이 있습니까?

In [ ]:
raw_bikes = pd.read_csv(raw_bikes_path(), encoding=ENCODING,
                        dtype=str, keep_default_na=False)
raw_rentals = pd.read_csv(raw_rentals_path(), encoding=ENCODING,
                        dtype=str, keep_default_na=False)
stations = pd.read_csv(stations_path(), encoding=ENCODING,
                        dtype=str, keep_default_na=False)

## STEP 2 · 자전거 마스터 정제

`raw-bikes.csv` 를 정제합니다.

### 요구사항

- **`station_id`** — 대문자로 통일, 빈 값은 결측으로 처리
- **`bike_type`** — `일반` / `전동` 두 가지로 통일
- **`gear_count`** — 단위 문자를 제거하고 정수로
- **`daily_fee`** — 콤마 제거 후 정수로
- **`manufacture_year`** — 정수로 (변환 실패는 결측으로)
- **중복 제거** — `bike_id` 기준

<aside>
💡

**전각 문자**

`unicodedata.normalize("NFKC", s)` 한 줄이면 전각 문자를 반각으로 통일할 수 있습니다.

전각 공백(`\u3000`)은 `strip()` 으로 지워지지 않으니 NFKC 를 먼저 적용하세요.

</aside>

### 기대 결과
<aside>
💡

**전각 문자**

`unicodedata.normalize("NFKC", s)` 한 줄이면 전각 문자를 반각으로 통일할 수 있습니다.

전각 공백(`\u3000`)은 `strip()` 으로 지워지지 않으니 NFKC 를 먼저 적용하세요.

</aside>

### 기대 결과

| 항목 | 값 |
| --- | --- |
| 행 수 | 50 |
| `bike_type` 고유값 | `['일반', '전동']` |
| `station_id` 고유값 | 25종 + 결측 1건 |
| `gear_count` 범위 | 3 ~ 7 |
| `daily_fee` 범위 | 1,000 ~ 2,000 |

## STEP 3 · 대여 기록 정제

`raw-rentals.csv` 를 정제합니다.

### 요구사항

- **`distance_km`, `fee`** — 콤마 제거 후 숫자로. 변환 실패는 결측으로
- **`rent_time`, `return_time`** — `datetime` 으로 통일 (형식 3종 혼재)
- **`payment_method`** — 공백 제거 후 대문자로 통일
- **중복 제거** — `rental_id` 기준

<aside>
💡

**콤마를 먼저 지우세요**

`"12,345"` 를 그냥 `to_numeric` 에 넣으면 멀쩡한 값이 결측이 됩니다.

</aside>

### 기대 결과
| 항목 | 값 |
| --- | --- |
| 중복 제거 후 행 수 | 14,500 |
| `distance_km` 결측 | 200 |
| `fee` 결측 | 0 |
| `payment_method` 고유값 | `['APP', 'CARD', 'MEMBERSHIP']` |

## STEP 4 · 결합

대여 기록에 자전거 정보와 대여소명·자치구를 붙입니다.

```
rentals ─(bike_id)→ bikes ─(station_id)→ stations
```

### 요구사항

- `how`, `validate` 를 명시할 것
- **`indicator=True` 로 매칭 실패를 확인**하고, 몇 건이 왜 실패했는지 설명할 것
- 매칭 실패한 행은 제외하고 진행

<aside>
⚠️

**행 수가 늘어나면 안 됩니다**

결합 전후 행 수를 반드시 비교하세요.

늘어났다면 오른쪽 테이블에 중복이 남아 있다는 뜻입니다. STEP 2 로 돌아가세요.

</aside>

### 확인 문항

1. 매칭에 실패한 `bike_id` 는 무엇입니까? 몇 건입니까?
2. 이런 데이터를 실무에서는 무엇이라고 부릅니까? 어떻게 처리해야 합니까?
3. 자치구명이 결측인 행이 있습니다. 원인은 무엇입니까?

### 기대 결과
| 항목 | 값 |
| --- | --- |
| 결합 후 행 수 | 14,500 (변화 없음) |
| 매칭 실패 | 80 건 |
| 제외 후 행 수 | 14,500 |
| 자치구 결측 | 304 |

## STEP 5 · 이상치와 논리 검사

**이 단계가 이 문제의 핵심입니다.**

통계 기법만으로는 찾을 수 없는 오류가 섞여 있습니다. 도메인 규칙으로 걸러내세요.

### 파생 컬럼

먼저 **대여 시간(분)** 을 계산합니다.

```
duration_min = (return_time - rent_time) 의 분 단위
```

### 검사 규칙

| 규칙 | 내용 | 근거 |
| --- | --- | --- |
| ① | 반납 시각이 대여 시각보다 앞서면 안 됨 | 물리적으로 불가능 |
| ② | 요금이 음수면 안 됨 | 논리적으로 불가능 |
| ③ | 이동 거리가 물리 상한을 넘으면 안 됨 | 자전거 속도의 한계 |

> 💡 **물리 상한을 어떻게 구합니까**
자전거로 시속 50km 이상을 지속하는 것은 불가능합니다.
`distance_km / (duration_min / 60)` 이 평균 속도(km/h)이고,
이 값이 **50km/h 를 초과**하면 이상치로 판정하세요.
> 

> ⚠️ **왜 IQR 로는 안 됩니까**
전동 자전거(빠름)와 일반 자전거(느림)가 섞여 있습니다.
일반 자전거에서 비정상적으로 부풀려진 거리값이, 전체 분포에서는 전동의 정상값처럼 보일 수 있습니다.
> 

### 기대 결과
| 규칙 | 탐지 건수 |
| --- | --- |
| ① 반납 ≤ 대여 | 50 |
| ② 요금 음수 | 20 |
| ③ 속도 > 50km/h | 650 |
| 합계 (중복 제외) | 719 |
| 제거 후 행 수 | 13,701 |

## STEP 6 · 결측 처리

### 요구사항

- **`fee` 결측** — 버리지 말고 복원하세요. `대여 시간(분) × 분당요금` 으로 계산할 수 있습니다
- **`distance_km` 결측** — 복원할 방법이 없으므로 해당 행을 제외

> 💡 **왜 요금은 복원하고 거리는 버립니까**
요금은 다른 컬럼으로부터 계산할 수 있지만, 이동 거리는 그럴 수 없기 때문입니다.
"채울 수 있는 근거가 있는가" 가 판단 기준입니다. 평균으로 때우는 것과는 다릅니다.
> 

### 기대 결과
| 항목 | 값 |
| --- | --- |
| 요금 복원 | 0 건 |
| 거리 결측 제거 | 200 건 |
| 최종 행 수 | 13,501 |

## STEP 7 · 집계

정제가 끝났는지 숫자로 확인합니다.

### 요구사항

1. 전체 요약 — 행 수, 자전거 수, 이용자 수, 총 이동 거리, 총 매출, 평균 대여 시간
2. 자치구별 집계 — 건수 · 이동 거리 · 매출 (매출 내림차순)
3. 자전거 타입별 집계 — 건수 · 평균 이동 거리 · 평균 대여 시간
4. 결제수단별 건수

### 검산 기준

아래 관계가 성립해야 합니다. 뒤집혔다면 어딘가에서 잘못 정제한 것입니다.

> 전동 자전거는 일반 자전거보다 **짧은 시간에 더 긴 거리**를 이동해야 합니다.
>

## STEP 8 · DB 적재

### 8-1. 스키마 설계

`rental_log` 테이블을 설계하고 생성하세요.

**설계 기준**

- 금액과 거리에 `FLOAT` 을 쓰지 말 것
- 날짜·시각을 문자열로 저장하지 말 것
- ID 는 계산하지 않는 값이므로 문자열로
- **`rental_id` 에 유니크 제약** — 재실행 안전성의 출발점

### 8-2. 적재

- `executemany` 또는 `to_sql(method="multi")` 로 벌크 적재
- **청크 단위로 커밋**
- `ON DUPLICATE KEY UPDATE` 로 UPSERT
- 신규 · 갱신 건수를 로그로 남길 것

### 8-3. 재실행 검증

**같은 스크립트를 두 번 실행하세요.**

|  | 1회차 | 2회차 |
| --- | --- | --- |
| 신규 | 13,501 | 0 |
| 갱신 | 0 | 0 |
| 최종 행 수 | 13,501 | 13,501 |

> ⚠️ **2회차에 행이 두 배가 되거나 에러가 났다면**
유니크 제약이 없거나, UPSERT 를 쓰지 않은 것입니다.
>

### 8-4. 적재 검증

행 수만 보면 놓칩니다. **집계값을 대조**하세요.

- 행 수 · 자전거 수 · 이용자 수
- 총 매출 · 총 이동 거리
- 최소 · 최대 날짜

> ⚠️ **합계가 미세하게 다르다면**
타입 변환에서 소수점이 잘렸을 가능성이 높습니다.
`DECIMAL` 자릿수와 파이썬 쪽 반올림을 맞췄는지 확인하세요.
>

## 결과물

| 파일 | 내용 |
| --- | --- |
| `pipeline/` 또는 `solution.py` | 전체 코드 |
| `report.md` | STEP 별 결과와 판단 근거 |
| `schema.sql` | 테이블 DDL |
| 실행 로그 | 단계별 건수가 보이는 것 |

### `report.md` 에 반드시 포함할 것

1. STEP 1 진단 결과 표
2. 각 STEP 의 처리 전후 행 수
3. **STEP 5 에서 왜 IQR 대신 도메인 규칙을 썼는지**
4. **STEP 6 에서 요금은 복원하고 거리는 버린 이유**
5. 재실행 검증 결과

## 심화 (여유가 있다면)

1. **ETL 분리** — `extract.py` / `transform.py` / `load.py` 로 나누고 `pipeline.py` 에서 조립
2. **로깅** — `logging` 으로 파일과 화면에 동시 기록. 단계별 건수를 남길 것
3. **검증 함수** — `validate()` 를 만들어 통과하지 못하면 적재하지 않도록
4. **시각화** — 자치구별 매출 막대그래프, 시간대별 이용 패턴 (한글 폰트 설정 포함)
5. **이용자 분석** — 재이용률, 이용자당 평균 대여 횟수